# 01 Data audit
Thin narrative over `src.services.quality`. Every value comes from `reports/dq_report.json`, produced by `sql/02_data_quality.sql` and `src/data/validation.py` in `make build`.

In [1]:
import logging, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.getLogger("streamlit").setLevel(logging.ERROR)
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)

## Data-quality rules
Each rule has a status. Problems are shown, not hidden.

In [2]:
from src.services import quality
checks = quality.checks()
print(quality.summary())
checks[['group', 'check', 'value', 'rule', 'status']]

2026-10-04 20:32:30.813 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-04 20:32:30.816 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-04 20:32:30.817 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


{'PASS': 19, 'WARNING': 7, 'FAIL': 0}


,group,check,value,rule,status
0,Volume,Application rows,438557.0,> 0,PASS
1,Volume,Credit rows,1048575.0,> 0,PASS
2,Volume,Unique applicants,438510.0,> 0,PASS
3,Volume,Unique credit clients,45985.0,> 0,PASS
4,Linkage,Matched clients (both files),36457.0,> 0,PASS
5,Linkage,Credit clients with demographics,0.792802,>= 75% else WARNING,PASS
6,Linkage,Applicants with credit history,0.083138,informational; low rate = WARNING,WARNING
7,Uniqueness,"Duplicate (ID, month) in credit file",0.0,= 0,PASS
8,Uniqueness,Duplicated application IDs,47.0,"= 0 else WARNING (kept once, flagged)",WARNING
9,Validity,Invalid status values,0.0,= 0,PASS


## Recomputed audit values vs the pilot targets in the brief

In [3]:
rec = quality.pilot_reconciliation()
print(f"{int(rec.agrees.sum())} of {len(rec)} items agree exactly")
rec

29 of 29 items agree exactly


,item,pilot,computed,difference,agrees
0,credit_rows,1048575.0,1048575.0,0.0,True
1,credit_clients,45985.0,45985.0,0.0,True
2,credit_dup_id_month,0.0,0.0,0.0,True
3,month_min,-60.0,-60.0,0.0,True
4,month_max,0.0,0.0,0.0,True
5,month_gap_pairs,0.0,0.0,0.0,True
6,matched_clients,36457.0,36457.0,0.0,True
7,application_only,402053.0,402053.0,0.0,True
8,credit_only,9528.0,9528.0,0.0,True
9,first_status_x_clients,18737.0,18737.0,0.0,True


## File truncation
The credit file has exactly Excel's row limit. The test checks whether the ID-sorted file is cut.

In [4]:
tr = quality.report()['truncation']
{k: tr[k] for k in ['row_count', 'excel_row_limit', 'app_ids_above_cut', 'app_ids_above_cut_matched', 'app_ids_at_or_below_cut', 'app_ids_at_or_below_cut_matched', 'final_client_first_status_is_c', 'fired', 'excluded']}

{'row_count': 1048575,
 'excel_row_limit': 1048575,
 'app_ids_above_cut': 402053.0,
 'app_ids_above_cut_matched': 0.0,
 'app_ids_at_or_below_cut': 36457.0,
 'app_ids_at_or_below_cut_matched': 36457.0,
 'final_client_first_status_is_c': True,
 'fired': True,
 'excluded': True}

## New charge-offs within 12 months by origin
These small counts shape the validation design.

In [5]:
pd.DataFrame(quality.report()['new_chargeoffs'])

,ORIGIN,open_0_4,new_co_12m
0,-36,5954,29
1,-30,6770,34
2,-24,7674,34
3,-18,8664,37
4,-12,9554,41
